In [41]:
# FINAL MODEL COMPARISON AT 1% FALSE-POSITIVE BUDGET

import pandas as pd
import json
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/intentmap-nids/intentmap-nids"
)

REPORT_DIR = PROJECT_ROOT / "reports"
CONFIG_DIR = PROJECT_ROOT / "config"
RESULT_DIR = PROJECT_ROOT / "results"

print("Report folder:", REPORT_DIR)
print("Config folder:", CONFIG_DIR)
print("Result folder:", RESULT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Report folder: /content/drive/MyDrive/intentmap-nids/intentmap-nids/reports
Config folder: /content/drive/MyDrive/intentmap-nids/intentmap-nids/config
Result folder: /content/drive/MyDrive/intentmap-nids/intentmap-nids/results


In [42]:
# MODEL RESULT FILES

budget_files = {
    "Isolation Forest": REPORT_DIR / "isolation_forest_budget_results.csv",
    "Autoencoder": REPORT_DIR / "autoencoder_budget_results.csv",
    "Hybrid IF+AE": PROJECT_ROOT / "results" / "hybrid" / "hybrid_budget_results.csv",
    "LOF": REPORT_DIR / "lof_budget_results.csv",
    "OCSVM": REPORT_DIR / "ocsvm_budget_results.csv",
    "Deep SVDD": REPORT_DIR / "deep_svdd_budget_results.csv"
}

config_files = {
    "Isolation Forest": CONFIG_DIR / "isolation_forest_candidate1.json",
    "Autoencoder": CONFIG_DIR / "autoencoder_candidate1.json",
    "Hybrid IF+AE": CONFIG_DIR / "hybrid_if_ae.json",
    "LOF": CONFIG_DIR / "lof_baseline.json",
    "OCSVM": CONFIG_DIR / "ocsvm_model.json",
    "Deep SVDD": CONFIG_DIR / "deep_svdd_model.json"
}

In [43]:
# BUILD FINAL 1% BUDGET COMPARISON

comparison_rows = []


# Convert values such as "4.82%" to 0.0482
def to_number(value):
    if pd.isna(value):
        return None

    if isinstance(value, str):
        value = value.strip()

        if value.endswith("%"):
            return float(value.replace("%", "")) / 100

    return float(value)


for model_name, budget_file in budget_files.items():

    if budget_file is None or not budget_file.exists():
        print("Missing budget file:", model_name)
        continue

    print("\nLoading:", model_name)

    results = pd.read_csv(budget_file)

    # Handle different F1 column names
    if "F1" not in results.columns:
        if "F1-score" in results.columns:
            results = results.rename(columns={"F1-score": "F1"})
        elif "F1 Score" in results.columns:
            results = results.rename(columns={"F1 Score": "F1"})

    # Find the 1% budget row
    budget_text = (
        results["Budget"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    one_percent_rows = results[
        budget_text.isin([
            "1% budget",
            "1.0% budget",
            "1%",
            "1.0%"
        ])
    ]

    if one_percent_rows.empty:
        print("No 1% row found for:", model_name)
        print("Available budgets:", results["Budget"].tolist())
        continue

    one_percent = one_percent_rows.iloc[0]

    # Get FPR
    if "Actual FPR" in results.columns:
        actual_fpr = to_number(one_percent["Actual FPR"])

    elif "FPR" in results.columns:
        actual_fpr = to_number(one_percent["FPR"])

    else:
        fp = int(one_percent["FP"])
        tn = int(one_percent["TN"])
        actual_fpr = fp / (fp + tn)

    # False alerts per 1000
    if "False alerts per 1000" in results.columns:
        false_alerts = to_number(
            one_percent["False alerts per 1000"]
        )

        # If accidentally stored as a percentage,
        # calculate it again from FPR
        if false_alerts is None:
            false_alerts = actual_fpr * 1000

    else:
        false_alerts = actual_fpr * 1000

    # Load ROC-AUC and PR-AUC from config
    roc_auc = None
    pr_auc = None

    config_file = config_files.get(model_name)

    if config_file is not None and config_file.exists():
        with open(config_file, "r") as f:
            config = json.load(f)

        roc_auc = config.get("roc_auc")
        pr_auc = config.get("pr_auc")

        if roc_auc is not None:
            roc_auc = to_number(roc_auc)

        if pr_auc is not None:
            pr_auc = to_number(pr_auc)

    comparison_rows.append({
        "Model": model_name,
        "Precision": to_number(one_percent["Precision"]),
        "Recall": to_number(one_percent["Recall"]),
        "F1": to_number(one_percent["F1"]),
        "Actual FPR": actual_fpr,
        "False alerts per 1000": actual_fpr * 1000,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "TN": int(one_percent["TN"]),
        "FP": int(one_percent["FP"]),
        "FN": int(one_percent["FN"]),
        "TP": int(one_percent["TP"])
    })


final_comparison = pd.DataFrame(comparison_rows)

display(final_comparison)


Loading: Isolation Forest

Loading: Autoencoder

Loading: Hybrid IF+AE

Loading: LOF

Loading: OCSVM

Loading: Deep SVDD


,Model,Precision,Recall,F1,Actual FPR,False alerts per 1000,ROC-AUC,PR-AUC,TN,FP,FN,TP
0,Isolation Forest,0.940079,0.617070,0.745072,0.048189,48.189189,0.851001,0.897443,35217,1783,17359,27973
1,Autoencoder,0.936372,0.632710,0.755157,0.052676,52.675676,0.878460,0.901098,35051,1949,16650,28682
2,Hybrid IF+AE,0.943977,0.625938,0.752742,0.045514,45.513514,0.876671,0.905663,35316,1684,16957,28375
3,LOF,0.936416,0.522721,0.670923,0.043486,43.486486,0.897508,0.888700,35391,1609,21636,23696
4,OCSVM,0.893863,0.339606,0.492207,0.049405,49.405405,0.794280,0.837517,35172,1828,29937,15395
5,Deep SVDD,0.927429,0.594547,0.724584,0.057000,57.000000,0.887058,0.895507,34891,2109,18380,26952


In [44]:
# FORMAT COMPARISON TABLE

display_table = final_comparison.copy()

for column in [
    "Precision",
    "Recall",
    "F1",
    "Actual FPR",
    "ROC-AUC",
    "PR-AUC"
]:
    display_table[column] = display_table[column].round(4)

display_table["False alerts per 1000"] = (
    display_table["False alerts per 1000"].round(2)
)

display(display_table)

,Model,Precision,Recall,F1,Actual FPR,False alerts per 1000,ROC-AUC,PR-AUC,TN,FP,FN,TP
0,Isolation Forest,0.9401,0.6171,0.7451,0.0482,48.19,0.8510,0.8974,35217,1783,17359,27973
1,Autoencoder,0.9364,0.6327,0.7552,0.0527,52.68,0.8785,0.9011,35051,1949,16650,28682
2,Hybrid IF+AE,0.9440,0.6259,0.7527,0.0455,45.51,0.8767,0.9057,35316,1684,16957,28375
3,LOF,0.9364,0.5227,0.6709,0.0435,43.49,0.8975,0.8887,35391,1609,21636,23696
4,OCSVM,0.8939,0.3396,0.4922,0.0494,49.41,0.7943,0.8375,35172,1828,29937,15395
5,Deep SVDD,0.9274,0.5945,0.7246,0.0570,57.00,0.8871,0.8955,34891,2109,18380,26952


In [45]:
# RANK MODELS AND BOLD THE BEST RESULT IN EACH COLUMN

ranked_models = final_comparison.sort_values(
    by="F1",
    ascending=False
).reset_index(drop=True)

ranked_models.insert(
    0,
    "Rank",
    range(1, len(ranked_models) + 1)
)

# Columns where HIGHER is better
higher_is_better = [
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "PR-AUC",
    "TN",
    "TP"
]

# Columns where LOWER is better
lower_is_better = [
    "Actual FPR",
    "False alerts per 1000",
    "FP",
    "FN"
]

styled_table = ranked_models.style.format({
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1": "{:.4f}",
    "Actual FPR": "{:.4%}",
    "False alerts per 1000": "{:.2f}",
    "ROC-AUC": "{:.4f}",
    "PR-AUC": "{:.4f}"
}, na_rep="-")

# Bold highest values
styled_table = styled_table.highlight_max(
    subset=higher_is_better,
    props="font-weight: bold;"
)

# Bold lowest values
styled_table = styled_table.highlight_min(
    subset=lower_is_better,
    props="font-weight: bold;"
)

display(styled_table)

,Rank,Model,Precision,Recall,F1,Actual FPR,False alerts per 1000,ROC-AUC,PR-AUC,TN,FP,FN,TP
0,1,Autoencoder,0.9364,0.6327,0.7552,5.2676%,52.68,0.8785,0.9011,35051,1949,16650,28682
1,2,Hybrid IF+AE,0.9440,0.6259,0.7527,4.5514%,45.51,0.8767,0.9057,35316,1684,16957,28375
2,3,Isolation Forest,0.9401,0.6171,0.7451,4.8189%,48.19,0.8510,0.8974,35217,1783,17359,27973
3,4,Deep SVDD,0.9274,0.5945,0.7246,5.7000%,57.00,0.8871,0.8955,34891,2109,18380,26952
4,5,LOF,0.9364,0.5227,0.6709,4.3486%,43.49,0.8975,0.8887,35391,1609,21636,23696
5,6,OCSVM,0.8939,0.3396,0.4922,4.9405%,49.41,0.7943,0.8375,35172,1828,29937,15395
